**PART 1.  THE CHAIN**

- Text -> Tokens -> Embeddings -> Attention -> Transformer -> Output

Day 19 gave you the first three links.  On Day 19 your AvgEmb model took the AVERAGE of the
word vectors.  Look back at EX 7(a): softmax of [0, 0, 0] = [1/3, 1/3, 1/3].  An average is
attention with EQUAL weights, so every word counts the same, whatever the sentence says.
A transformer lets the weights DEPEND on the words: in "the bank of the river", the vector
of "bank" can take most of its weight from "river".
 
Three limits of Day 19 models that the transformer repairs:
  - Bag of Words and the average: word order and context are gone.
  - The weights of every word are fixed (equal), not decided by the sentence.
  - A word has ONE vector, whatever it means in the sentence.

**PART 2.  SELF-ATTENTION:  QUERY, KEY, VALUE**

- Picture a library.  A token asks a question (QUERY).  Every token has a label on its
spine (KEY) and its content inside (VALUE).  The token compares its query with every
key; the better a key matches, the more of that token's value it reads.

How they are made: three weight matrices W_Q, W_K, W_V (LEARNED by gradient descent,
like every weight since Day 14) turn each token's embedding x into three vectors:
 
        q = x W_Q          k = x W_K          v = x W_V
 
For all n tokens at once, X has one row per token:
 
        Q = X W_Q ,   K = X W_K ,   V = X W_V
 
        Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V
 
   shapes:  X (n, d_model) ;  W_Q, W_K (d_model, d_k) ;  W_V (d_model, d_v)
            Q, K (n, d_k) ;  V (n, d_v) ;  scores Q K^T (n, n) ;  output (n, d_v)

Four steps for each token:
   1. score against every token: the dot product q . k     (the (n, n) table)
   2. divide by sqrt(d_k)                                  (Part 3)
   3. softmax each ROW: weights >= 0 and add up to 1       (Day 19 Part 6)
   4. new vector = the weighted sum of the VALUES
 
"Self"-attention: Q, K and V all come from the same sequence.
 
THE WORKED EXAMPLE  (use it in EX 1 to EX 3).  d_model = 2, d_k = 2, three tokens:
 
        X = [[1, 0],        W_Q = [[1, 1],      W_K = [[1, 0],      W_V = [[1, 0],
             [0, 1],               [0, 1]]             [1, 1]]             [0, 2]]
             [1, 1]]
 
(Row i of X is token i.  Row of a matrix times a matrix: for x = [1, 0] and W_Q the result
is the first row of W_Q, [1, 1].)

- EX 1 (paper, 10 min).  Compute Q, K and V (each is 3 x 2).
  * Q = X W_Q = [[1,1], [0,1], [1,2]]
  * K = X W_K = [[1,0], [1,1], [2,1]]
  * V = X W_V = [[1,0], [0,2], [1,2]]
- EX 2 (paper, 10 min).  Compute the raw score table S = Q K^T (3 x 3; the entry in row i
  and column j is q_i . k_j), and then S / sqrt(2) (give 3 decimals).
  * S = Q K^T = [[1, 2, 3],
                 [0, 1, 1],
                 [1, 3, 4]]
  * S / sqrt(2) = [[0.707, 1.414, 2.121],
                   [0.000, 0.707, 0.707],
                   [0.707, 2.121, 2.828]]
        
- EX 3 (paper, 15 min).  Token 3 only:

     a) softmax of row 3 of the scaled table.  Check that the three weights add up to 1.
      * row 3 = [0.707, 2.121, 2.828]
      * expo of row 3 = 2.028, 8.339, 16.911
      * row 3 / total expo = 0.074, 0.306, 0.620 = 1.0

     b) the new vector of token 3 = weights x the rows of V.
      * 0.074 x [1, 0] = [0.074, 0.000], 0.306 x [0, 2] = [0.0, 0.612], 0.620 x [1, 2] = [0.620, 1.240]
      * new vector: [0.694, 1.852]

     c) which token does token 3 pay the most attention to, and the least?
      * Token 3 pays the most attention to Token 3 (itself), with the highest weight of 0.620.
      Token 3 pays the least attention to Token 1, with the lowest weight of 0.074.
      
  (Rows 1 and 2 you will check with code in EX 8.)

**PART 3.  WHY DIVIDE BY sqrt(d_k)?**

If q and k have d_k numbers each, their dot product is a sum of d_k products, so it GROWS
with d_k (for random vectors its spread is about sqrt(d_k): 2 for d_k = 4, 8 for 64, 22.6
for 512).  Large scores make the softmax almost one-hot (one weight near 1, the rest near
0), and then its gradient is nearly zero, so learning stalls.  Dividing by sqrt(d_k) keeps
the spread of the scores around 1, whatever the size.
 
  EX 4 (paper, 10 min).

  a) softmax([4, 0, 0])  and  softmax([2, 0, 0]).  (The second is the first divided by
   sqrt(d_k) = 2, i.e. d_k = 4.)
  * expo of softmax([4,0,0]): e^4 = 54.60, e^0 = 1 
  * sum of expo / expo of softmax = [0.965, 0.018, 0.018]

  * expo of softmax([2, 0, 0]): e^2 = 7.39, e^0 = 1
  * sum of expo / expo of softmax = [0.787, 0.106, 0.106]

  b) In one sentence: what happens to the weights without the division when d_k is 64?
  * Weights without the division when d_k is 64 the raw dot product scores become massive, driving softmax into an extreme "one-hot" state (where one weight is basically $1.0$ and the rest are $0.0$), which flattens the gradient to zero and stops the neural network from learning.

  c) Your StandardScaler (Day 8, and the z-scores of Day F1) also keeps numbers in a
     reasonable range before they enter a model.  Name the common idea.
  * Keep the numbers in range where the model will behave well on normalising and scaling.